# AI Energy Consumption Optimizer
## Phase 5: Anomaly Detection (Multi-Condition AND-Gate)

### Design Rationale
Previous versions suffered from precision collapse because any single gate firing (OR-logic) would trigger an alert, flooding the system with false positives from natural energy variance.

This version implements a **strict multi-condition AND-gate** that requires ALL three statistical conditions to be simultaneously true, plus the Isolation Forest must agree:

```
stat_flag = (
    value > rolling_mean + 3.5*std  # Extreme statistical deviation
    AND value > 1.5 kWh             # Absolute minimum threshold
    AND (value / rolling_mean) > 2  # Relative spike (at least 2x baseline)
)
combined_flag = stat_flag AND iso_flag
```

**Why these three conditions together?**
- `3.5σ` alone misses near-zero-baseline spikes (e.g., 0.5 kWh at 3 AM)
- The `1.5 kWh` absolute floor ensures we only care about meaningful energy draws
- The `ratio > 2` ensures the spike is at least **double** the recent baseline — not just normal morning load ramp-up

In [ ]:
from google.colab import drive
import os
import pandas as pd
import numpy as np
import joblib
import json
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import IsolationForest
from sklearn.metrics import precision_score, recall_score, f1_score

drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/energy-optimizer'

Mounted at /content/drive


In [ ]:
# 1. Reload and prepare hourly data
RAW_PATH = os.path.join(BASE, 'data/raw/household_power_consumption.csv')
df = pd.read_csv(RAW_PATH, sep=';', na_values=['?'], dtype={'Global_active_power': 'float64'})
df['Datetime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'], format='%d/%m/%Y %H:%M:%S')
df.set_index('Datetime', inplace=True)
df.drop(columns=['Date', 'Time'], inplace=True)

df_hourly = df[['Global_active_power']].resample('1H').mean()
df_hourly.rename(columns={'Global_active_power': 'consumption'}, inplace=True)
df_hourly['consumption'] = df_hourly['consumption'].ffill(limit=3)
df_hourly['consumption'] = df_hourly['consumption'].interpolate(method='linear', limit=24)
df_hourly.dropna(inplace=True)
print(f"Clean hourly shape: {df_hourly.shape}")

Clean hourly shape: (34345, 1)


In [ ]:
# 2. Build 6-feature anomaly matrix
df_hourly['rolling_mean_24h'] = df_hourly['consumption'].shift(1).rolling(window=24).mean()
df_hourly['rolling_std_24h'] = df_hourly['consumption'].shift(1).rolling(window=24).std()
df_hourly['hour_of_day'] = df_hourly.index.hour
df_hourly['day_of_week'] = df_hourly.index.dayofweek
df_hourly['residual'] = df_hourly['consumption'] - df_hourly['rolling_mean_24h']
df_hourly.dropna(inplace=True)

ANOMALY_FEATURES = ['consumption', 'rolling_mean_24h', 'rolling_std_24h', 'hour_of_day', 'day_of_week', 'residual']
print(f"Feature matrix: {df_hourly[ANOMALY_FEATURES].shape}")

Feature matrix: (34321, 6)


In [ ]:
# 3. Temporal split
n = len(df_hourly)
train_end = int(n * 0.70)

X_train_anom = df_hourly[ANOMALY_FEATURES].iloc[:train_end]
X_test_anom  = df_hourly[ANOMALY_FEATURES].iloc[train_end:]

print(f"Train: {X_train_anom.shape}, Test: {X_test_anom.shape}")

Train: (24024, 6), Test: (10297, 6)


In [ ]:
# 4. Train Isolation Forest (tuned for precision)
iso_forest = IsolationForest(
    contamination=0.01,
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)
iso_forest.fit(X_train_anom)

model_path = os.path.join(BASE, 'artifacts/models/isolation_forest.pkl')
joblib.dump(iso_forest, model_path)
print(f"✅ Isolation Forest trained (contamination=0.01) and saved.")

✅ Isolation Forest trained (contamination=0.01) and saved.


In [ ]:
# 5. Inject 30 synthetic spikes (ADDITIVE — mathematically valid against any baseline)
np.random.seed(42)
X_test_modified = X_test_anom.copy()

n_inject = 30
inject_indices = np.random.choice(len(X_test_modified), size=n_inject, replace=False)

ground_truth = np.zeros(len(X_test_modified), dtype=int)
ground_truth[inject_indices] = 1

# Additive spikes: +3.0 to +5.0 kWh on top of whatever the baseline is
spike_additions = np.random.uniform(3.0, 5.0, size=n_inject)
for i, idx in enumerate(inject_indices):
    X_test_modified.iloc[idx, X_test_modified.columns.get_loc('consumption')] += spike_additions[i]
    X_test_modified.iloc[idx, X_test_modified.columns.get_loc('residual')] = (
        X_test_modified.iloc[idx, X_test_modified.columns.get_loc('consumption')]
        - X_test_modified.iloc[idx, X_test_modified.columns.get_loc('rolling_mean_24h')]
    )

print(f"Injected {n_inject} synthetic anomalies (+3.0 to +5.0 kWh additive).")

Injected 30 synthetic anomalies (+3.0 to +5.0 kWh additive).


In [ ]:
# 6. Multi-Condition Statistical Gate
cons   = X_test_modified['consumption']
mean24 = X_test_modified['rolling_mean_24h']
std24  = X_test_modified['rolling_std_24h']

# Condition A: 3.5-sigma deviation
cond_sigma   = cons > (mean24 + 3.5 * std24)

# Condition B: Absolute minimum meaningful energy draw
cond_abs     = cons > 1.5

# Condition C: Relative spike — at least 2x the rolling baseline
# Guard against division by zero (if mean is 0 or very small)
safe_mean    = mean24.clip(lower=0.01)
ratio        = cons / safe_mean
cond_ratio   = ratio > 2.0

# Combined statistical flag — ALL three must be true
stat_flags   = (cond_sigma & cond_abs & cond_ratio).astype(int)

# ML Gate
iso_preds    = iso_forest.predict(X_test_modified)
iso_flags    = (iso_preds == -1).astype(int)

# STRICT AND-GATE
combined_flags = ((stat_flags == 1) & (iso_flags == 1)).astype(int)

precision = precision_score(ground_truth, combined_flags, zero_division=0)
recall    = recall_score(ground_truth, combined_flags, zero_division=0)
f1        = f1_score(ground_truth, combined_flags, zero_division=0)

false_alarms_idx = np.where((ground_truth == 0) & (combined_flags == 1))[0]
missed_idx       = np.where((ground_truth == 1) & (combined_flags == 0))[0]

print("\n" + "="*52)
print("   FINAL ANOMALY DETECTION REPORT")
print("="*52)
print(f"  Precision:        {precision:.3f}  (Target > 0.70)")
print(f"  Recall:           {recall:.3f}  (Target > 0.65)")
print(f"  F1 Score:         {f1:.3f}  (Target > 0.72)")
print(f"  Total flags:      {combined_flags.sum()}")
print(f"  False alarms:     {len(false_alarms_idx)}")
print(f"  Missed anomalies: {len(missed_idx)}")
print("="*52)

print(f"\n  Stat gate fired:  {stat_flags.sum()}")
print(f"  ISO gate fired:   {iso_flags.sum()}")

if precision > 0.70 and recall > 0.65 and f1 > 0.72:
    print("\n✅ ALL TARGETS MET — System is production-ready.")
else:
    print("\n⚠️ WARNING: One or more targets not met.")
    if precision < 0.70:
        print("   → Precision low: lower contamination (0.01 → 0.005) OR tighten ratio to 2.5")
    if recall < 0.65:
        print("   → Recall low: lower sigma (3.5 → 3.0) OR lower ratio (2.0 → 1.5)")


   FINAL ANOMALY DETECTION REPORT
  Precision:        0.846  (Target > 0.70)
  Recall:           0.733  (Target > 0.65)
  F1 Score:         0.786  (Target > 0.72)
  Total flags:      26
  False alarms:     4
  Missed anomalies: 8

  Stat gate fired:  116
  ISO gate fired:   43

✅ ALL TARGETS MET — System is production-ready.


In [ ]:
# 7. Diagnostic — Inspect False Positives and Misses
print("\n--- DIAGNOSTIC: Sample False Positives (Top 5) ---")
for idx in false_alarms_idx[:5]:
    c = X_test_modified.iloc[idx]['consumption']
    m = X_test_modified.iloc[idx]['rolling_mean_24h']
    s = X_test_modified.iloc[idx]['rolling_std_24h']
    r = c / max(m, 0.01)
    sigma_away = (c - m) / s if s > 0 else 0
    print(f"  idx {idx}: value={c:.2f}, mean={m:.2f}, std={s:.2f}, sigma={sigma_away:.1f}, ratio={r:.1f}")
if len(false_alarms_idx) == 0:
    print("  ✅ No false positives!")

print("\n--- DIAGNOSTIC: Missed Anomalies ---")
for idx in missed_idx:
    c = X_test_modified.iloc[idx]['consumption']
    m = X_test_modified.iloc[idx]['rolling_mean_24h']
    s = X_test_modified.iloc[idx]['rolling_std_24h']
    r = c / max(m, 0.01)
    sigma_away = (c - m) / s if s > 0 else 0
    print(f"  idx {idx}: value={c:.2f}, mean={m:.2f}, std={s:.2f}, sigma={sigma_away:.1f}, ratio={r:.1f}")
if len(missed_idx) == 0:
    print("  ✅ Zero misses!")


--- DIAGNOSTIC: Sample False Positives (Top 5) ---
  idx 2318: value=4.21, mean=1.06, std=0.77, sigma=4.1, ratio=4.0
  idx 3637: value=4.62, mean=1.16, std=0.82, sigma=4.2, ratio=4.0
  idx 3772: value=5.09, mean=1.17, std=0.86, sigma=4.6, ratio=4.3
  idx 10149: value=5.63, mean=1.18, std=0.71, sigma=6.2, ratio=4.7

--- DIAGNOSTIC: Missed Anomalies ---
  idx 33: value=4.23, mean=1.05, std=0.83, sigma=3.8, ratio=4.0
  idx 318: value=3.42, mean=0.84, std=0.74, sigma=3.5, ratio=4.1
  idx 1199: value=4.43, mean=1.27, std=0.90, sigma=3.5, ratio=3.5
  idx 3437: value=4.30, mean=1.75, std=1.17, sigma=2.2, ratio=2.5
  idx 3688: value=4.38, mean=1.43, std=0.97, sigma=3.0, ratio=3.1
  idx 6202: value=3.41, mean=0.70, std=0.44, sigma=6.2, ratio=4.9
  idx 6662: value=3.70, mean=1.13, std=0.75, sigma=3.5, ratio=3.3
  idx 9489: value=3.39, mean=1.79, std=0.79, sigma=2.0, ratio=1.9


In [ ]:
# 8. Save anomaly features and update metadata
with open(os.path.join(BASE, 'artifacts/scalers/anomaly_feature_columns.json'), 'w') as f:
    json.dump(ANOMALY_FEATURES, f)

metadata_path = os.path.join(BASE, 'artifacts/metadata.json')
with open(metadata_path) as f:
    metadata = json.load(f)

metadata['anomaly_model_filename'] = 'isolation_forest.pkl'
metadata['anomaly_features'] = 'anomaly_feature_columns.json'
metadata['anomaly_config'] = {
    'gate_logic': 'AND',
    'contamination': 0.01,
    'sigma_threshold': 3.5,
    'abs_threshold_kwh': 1.5,
    'ratio_threshold': 2.0
}
metadata['anomaly_metrics'] = {
    'precision': round(precision, 3),
    'recall': round(recall, 3),
    'f1': round(f1, 3)
}

with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=4)

print("\n⭐ Phase 5 Complete. Updated metadata.json:")
print(json.dumps(metadata, indent=4))


⭐ Phase 5 Complete. Updated metadata.json:
{
    "version_tag": "Production-v1",
    "algorithm": "LightGBM",
    "model_filename": "forecast_winner.pkl",
    "q05_filename": "xgb_q05.pkl",
    "q95_filename": "xgb_q95.pkl",
    "scaler_filename": "main_scaler.pkl",
    "feature_columns": "feature_columns.json",
    "metrics": {
        "test_rmse": 0.5035,
        "test_mae": 0.3483
    },
    "anomaly_model_filename": "isolation_forest.pkl",
    "anomaly_features": "anomaly_feature_columns.json",
    "anomaly_metrics": {
        "precision": 0.846,
        "recall": 0.733,
        "f1": 0.786
    },
    "anomaly_config": {
        "gate_logic": "AND",
        "contamination": 0.01,
        "sigma_threshold": 3.5,
        "abs_threshold_kwh": 1.5,
        "ratio_threshold": 2.0
    }
}
